# Day 14 — Expression expansion: one expression, many columns

*100 Days of Polars · Phase 2: Expressions*

- **Post:** [Expression Expansion: One Expression, Many Columns](https://python.plainenglish.io/day-14-expression-expansion-one-expression-many-columns-181a7bcd5f1a)
- **Dataset:** [Olist Brazilian e-commerce](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) — CSVs in `data/raw/`
- **Written for:** Polars 2.x (run from the repo root)

In [1]:
import polars as pl

print("polars", pl.__version__)

polars 2.0.0


One expression can stand for **many** columns. Polars expands it into one expression per column before running anything.
`pl.col("a", "b").sum()` is shorthand for `pl.col("a").sum(), pl.col("b").sum()`.

In [2]:
items = pl.read_csv("data/raw/olist_order_items_dataset.csv", try_parse_dates=True)
products = pl.read_csv("data/raw/olist_products_dataset.csv")

## 1. Multiple names expand

In [3]:
items.select(pl.col("price", "freight_value").sum())

price,freight_value
f64,f64
1.3592e7,2.2519e6


In [4]:
items.group_by("seller_id").agg(pl.col("price", "freight_value").sum()).head(3)

seller_id,price,freight_value
str,f64,f64
"""06df0c10d34e459d9672bc77388742…",9070.0,525.77
"""89ad3263c0931c47f976d7c5778e60…",568.2,131.54
"""446cc09cdbb130904da114e807dbec…",252.8,54.17


## 2. `pl.all()` and `exclude`

In [5]:
# A null profile of the whole table in one line
products.select(pl.all().null_count())

product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
u32,u32,u32,u32,u32,u32,u32,u32,u32
0,610,610,610,610,2,2,2,2


In [6]:
products.select(pl.all().n_unique())

product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
u32,u32,u32,u32,u32,u32,u32,u32,u32
32951,74,67,2961,20,2205,100,103,96


In [7]:
# Everything except the ids
products.select(pl.all().exclude("product_id").null_count())

product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
u32,u32,u32,u32,u32,u32,u32,u32
610,610,610,610,2,2,2,2


In [8]:
# Exclude takes names/regex OR dtypes in one call, not both:
try:
    products.select(pl.exclude("^product_.*_lenght$", pl.String))
except TypeError as err:
    print("TypeError ->", err)

# Fix a) chain two excludes   b) selectors, which mix freely
import polars.selectors as cs
a = products.select(pl.exclude("^product_.*_lenght$").exclude(pl.String)).columns
b = products.select(cs.all() - cs.matches("_lenght$") - cs.string()).columns
print("a)", a)
print("b)", b)

TypeError -> cannot exclude by both column name and dtype
a) ['product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
b) ['product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']


## 3. Regex and dtype expansion

In [9]:
products.select(pl.col("^product_.*_cm$").max())

product_length_cm,product_height_cm,product_width_cm
i64,i64,i64
105,105,118


In [10]:
items.with_columns(pl.col(pl.Float64).round(0)).head(3)

order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
str,i64,str,str,datetime[μs],f64,f64
"""00010242fe8c5a6d1ba2dd792cb162…",1,"""4244733e06e7ecb4970a6e2683c13e…","""48436dade18ac8b2bce089ec2a0412…",2017-09-19 09:45:35,59.0,13.0
"""00018f77f2f0320c557190d7a144bd…",1,"""e5f2d52b802189ee658865ca93d83a…","""dd7ddc04e1b6c2c614352b383efe2d…",2017-05-03 11:05:13,240.0,20.0
"""000229ec398224ef6ca0657da4fc70…",1,"""c777355d18b72b67abbeef9df44fd0…","""5b51032eddd242adc84c38acab88f2…",2018-01-18 14:48:30,199.0,18.0


## 4. Across columns instead of down them

In [11]:
products.select(
    "product_id",
    total_dimensions_cm=pl.sum_horizontal(pl.col("^product_.*_cm$")),
    longest_side_cm=pl.max_horizontal(pl.col("^product_.*_cm$")),
).head(5)

product_id,total_dimensions_cm,longest_side_cm
str,i64,i64
"""1e9e8ef04dbcff4541ed26657ea517…",40,16
"""3aa071139cb16b67ca9e5dea641aaa…",68,30
"""96bd76ec8810374ed1b65e29197571…",42,18
"""cef67bcfe19066a932b7673e239eb2…",56,26
"""9dc1a7de274444849c219cff195d0b…",50,20


## 5. Gotcha: expansion meets duplicate names

Each expanded expression keeps its column's name. Two aggregations of the same column collide.

In [12]:
try:
    items.select(pl.col("price").sum(), pl.col("price").mean())
except Exception as err:
    print(type(err).__name__, "->", str(err).splitlines()[0])

DuplicateError -> projections contained duplicate output name 'price'. It's possible that multiple expressions are returning the same default column name. If this is the case, try renaming the columns with `.alias("new_name")` to avoid duplicate column names.


In [13]:
# Fix: give each output its own name
items.select(price_sum=pl.col("price").sum(), price_mean=pl.col("price").mean())

price_sum,price_mean
f64,f64
1.3592e7,120.653739


In [14]:
# Can you tell before running that an expression will expand?
print(pl.col("price", "freight_value").meta.has_multiple_outputs())
print(pl.col("price").meta.has_multiple_outputs())
print(pl.all().meta.has_multiple_outputs())

True
False
True


In [15]:
# alias on an expanded expression: every output gets the SAME name -> collision
try:
    items.select(pl.col("price", "freight_value").sum().alias("total"))
except Exception as err:
    print(type(err).__name__, "->", str(err).splitlines()[0])

DuplicateError -> projections contained duplicate output name 'total'. It's possible that multiple expressions are returning the same default column name. If this is the case, try renaming the columns with `.alias("new_name")` to avoid duplicate column names.


Naming many expanded outputs at once (suffixes, prefixes, a rename function) is the next topic.

## Try it yourself

1. Min and max of every numeric column of `products`, in one `select`. (You will hit the duplicate-name error. Keep the error; fix it in the next notebook.)
2. Count distinct values of every String column of `items`.
3. Use `pl.mean_horizontal` to average the three `_cm` dimensions per product.